<a href="https://colab.research.google.com/github/NataliiaFakas/8-puzzle-toolkit/blob/main/5_RF_variaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Enfoque A — modelo actual, pero corregido

Mantienes las ventas absolutas y las variables macro actuales, pero:

Partición temporal:
Train: 2005–2018
Dev: 2019–2021
Test: 2022–2025
Sin shuffle.
Crear un baseline naive.
Comparar:
Baseline
Random Forest
Usar MSE y RMSE.
Enfoque B — nueva transformación de los datos

Crear una versión nueva donde:

Eliminas 2005.
La variable objetivo deja de ser ventas absolutas.
Pasas a predecir la variación porcentual de las matriculaciones.
Las variables macroeconómicas absolutas pasan a ser variaciones relativas.
Mantienes como están las variables que ya son tasas/porcentajes:
GDP_Growth
Retail_Sales_Growth
Unemployment_Rate
Repites:
Partición temporal estática.
Baseline.
Random Forest.
TimeSeriesSplit con 4 bloques.
Comparas los resultados finales con el enfoque anterior.

La idea final es averiguar si trabajar con variaciones relativas mejora la capacidad predictiva del Random Forest.

## **Construcción del dataset definitivo**

---

# Introducción

A lo largo de este cuaderno, se construirá el dataset definitivo mediante la función `añadir_variables_macro`, que combinará las ventas anuales de BMW con las variables macroeconómicas `Compensation_Per_Employee`, `Industrial_Production`, `Population_Total` y `EUR_USD` seleccionadas en el análisis de importancia utilizando Random Forest.

El resultado obtenido será un conjunto de datos de 21 observaciones (2005-2025) y 6 columnas (año, variable objetivo y cuatro predictores), que constituye la base sobre la que se entrenará y evaluará el modelo predictivo.

In [1]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

ruta_macro = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_variables_macro.csv"
ruta_bmw = "/content/drive/MyDrive/TFG_BMW/datasets/ventas_bmw.csv"

df_macro = pd.read_csv(ruta_macro)
df_bmw = pd.read_csv(ruta_bmw)

display(df_macro.head())
display(df_bmw.head())

Mounted at /content/drive


,Year,Unemployment_Rate,GDP_Growth,Deposit_Facility,HICP,Industrial_Production,Retail_Sales_Growth,Consumer_Confidence,Compensation_Per_Employee,Brent_Oil_Price,EUR_USD,Euribor_12M,Population_Total
0,2005,8.9,1.7,1.25,2.18,153.2,0.03,-4.2,31.22,54.57,1.2441,2.19,434585887
1,2006,8.2,3.2,2.50,2.19,147.7,0.04,-3.5,31.95,65.16,1.2556,3.08,436041018
2,2007,7.2,2.9,3.00,2.13,145.5,0.03,-2.1,32.75,72.44,1.3705,4.24,437514477
3,2008,7.1,0.4,2.00,3.30,148.0,0.01,-10.8,33.86,96.94,1.4708,4.63,439074280
4,2009,9.0,-4.4,0.25,0.29,144.5,-0.03,-21.7,34.41,61.74,1.3948,1.31,440467898


,Año,Ventas_BMW_Unidades
0,2005,1126768
1,2006,1185088
2,2007,1276793
3,2008,1202239
4,2009,1068770


# Función `variables_modelo`

---

🔧 **Reutilizaremos esta función cada vez que se entrene un nuevo modelo para incorporar al conjunto de datos de ventas de BMW**, las variables macroeconómicas seleccionadas utilizando el año como clave de unión.

In [2]:
def añadir_variables_macro(df_bmw, df_macro):

    df_final = df_bmw.merge(
        df_macro[variables_modelo],
        on="Year",
        how="left"
    )

    return df_final

In [3]:
variables_modelo = [
    "Year",
    "Compensation_Per_Employee",
    "Industrial_Production",
    "Population_Total",
    "EUR_USD"
]

df_macro_reducido = df_macro[variables_modelo]

df_macro_reducido.head()

,Year,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD
0,2005,31.22,153.2,434585887,1.2441
1,2006,31.95,147.7,436041018,1.2556
2,2007,32.75,145.5,437514477,1.3705
3,2008,33.86,148.0,439074280,1.4708
4,2009,34.41,144.5,440467898,1.3948


In [4]:
df_bmw_filtrado = df_bmw.rename(columns={"Año": "Year"})

data = añadir_variables_macro(df_bmw_filtrado, df_macro_reducido)

display(data.head())
print("Shape del dataset definitivo:", data.shape)

,Year,Ventas_BMW_Unidades,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD
0,2005,1126768,31.22,153.2,434585887,1.2441
1,2006,1185088,31.95,147.7,436041018,1.2556
2,2007,1276793,32.75,145.5,437514477,1.3705
3,2008,1202239,33.86,148.0,439074280,1.4708
4,2009,1068770,34.41,144.5,440467898,1.3948


Shape del dataset definitivo: (21, 6)


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42  # semilla fija para reproducibilidad

X = data.drop(columns=["Year", "Ventas_BMW_Unidades"])
y = data["Ventas_BMW_Unidades"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (21, 4)
y shape: (21,)


# PARTE 1: Eliminamos la partición aleatoria

Eliminamos esta celda:

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    shuffle=True
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    shuffle=True
)

#y la sustituimos por una partición temporal

In [6]:
# Partición temporal:
# Train: 2005-2018
# Dev:   2019-2021
# Test:  2022-2025

train = data[data["Year"] <= 2018]
dev = data[(data["Year"] >= 2019) & (data["Year"] <= 2021)]
test = data[data["Year"] >= 2022]

X_train = train.drop(columns=["Year", "Ventas_BMW_Unidades"])
y_train = train["Ventas_BMW_Unidades"]

X_dev = dev.drop(columns=["Year", "Ventas_BMW_Unidades"])
y_dev = dev["Ventas_BMW_Unidades"]

X_test = test.drop(columns=["Year", "Ventas_BMW_Unidades"])
y_test = test["Ventas_BMW_Unidades"]

print(f"Train: {len(train)} muestras ({train['Year'].min()}-{train['Year'].max()})")
print(f"Dev:   {len(dev)} muestras ({dev['Year'].min()}-{dev['Year'].max()})")
print(f"Test:  {len(test)} muestras ({test['Year'].min()}-{test['Year'].max()})")

Train: 14 muestras (2005-2018)
Dev:   3 muestras (2019-2021)
Test:  4 muestras (2022-2025)


# PARTE 2 — Crear el baseline naive

In [7]:
data_baseline = data.copy()

data_baseline["Prediccion_Naive"] = (
    data_baseline["Ventas_BMW_Unidades"].shift(1)
)

# Para 2005 no existe un año anterior
data_baseline.loc[
    data_baseline["Year"] == 2005,
    "Prediccion_Naive"
] = 0

display(data_baseline.head())

,Year,Ventas_BMW_Unidades,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD,Prediccion_Naive
0,2005,1126768,31.22,153.2,434585887,1.2441,0.0
1,2006,1185088,31.95,147.7,436041018,1.2556,1126768.0
2,2007,1276793,32.75,145.5,437514477,1.3705,1185088.0
3,2008,1202239,33.86,148.0,439074280,1.4708,1276793.0
4,2009,1068770,34.41,144.5,440467898,1.3948,1202239.0


# PARTE 3 — Evaluar el baseline

In [8]:
from sklearn.metrics import mean_squared_error

pred_naive_dev = data_baseline.loc[
    data_baseline["Year"].between(2019, 2021),
    "Prediccion_Naive"
]

y_naive_dev = data_baseline.loc[
    data_baseline["Year"].between(2019, 2021),
    "Ventas_BMW_Unidades"
]

mse_naive_dev = mean_squared_error(
    y_naive_dev,
    pred_naive_dev
)

rmse_naive_dev = np.sqrt(mse_naive_dev)

print(f"MSE baseline en dev: {mse_naive_dev:,.2f}")
print(f"RMSE baseline en dev: {rmse_naive_dev:,.2f}")

MSE baseline en dev: 18,575,566,348.33
RMSE baseline en dev: 136,292.21


In [9]:
pred_naive_test = data_baseline.loc[
    data_baseline["Year"] >= 2022,
    "Prediccion_Naive"
]

y_naive_test = data_baseline.loc[
    data_baseline["Year"] >= 2022,
    "Ventas_BMW_Unidades"
]

mse_naive_test = mean_squared_error(
    y_naive_test,
    pred_naive_test
)

rmse_naive_test = np.sqrt(mse_naive_test)

print(f"MSE baseline en test: {mse_naive_test:,.2f}")
print(f"RMSE baseline en test: {rmse_naive_test:,.2f}")

MSE baseline en test: 10,012,345,269.50
RMSE baseline en test: 100,061.71


# PARTE 4 — Random Forest con partición temporal

In [10]:
n_estimators_grid = [10, 25, 50, 100, 200, 300]
min_impurity_decrease_grid = [0.0, 1e6, 5e6, 1e7, 5e7, 1e8]

resultados = []

for n_est in n_estimators_grid:
    for mid in min_impurity_decrease_grid:

        modelo = RandomForestRegressor(
            n_estimators=n_est,
            min_impurity_decrease=mid,
            random_state=RANDOM_STATE
        )

        modelo.fit(X_train, y_train)

        pred_dev = modelo.predict(X_dev)

        mse_dev = mean_squared_error(
            y_dev,
            pred_dev
        )

        resultados.append({
            "n_estimators": n_est,
            "min_impurity_decrease": mid,
            "MSE_dev": mse_dev
        })

df_resultados = (
    pd.DataFrame(resultados)
    .sort_values("MSE_dev")
)

display(df_resultados.head(10))

,n_estimators,min_impurity_decrease,MSE_dev
12,50,0.0,7.479421e+09
13,50,1000000.0,7.479421e+09
14,50,5000000.0,7.479421e+09
15,50,10000000.0,7.479421e+09
24,200,0.0,7.616778e+09
25,200,1000000.0,7.616778e+09
26,200,5000000.0,7.616778e+09
27,200,10000000.0,7.616778e+09
16,50,50000000.0,7.732620e+09
6,25,0.0,7.856447e+09


# PARTE 5 — Evaluar este Random Forest en test

In [11]:
mejor = df_resultados.iloc[0]

modelo_final = RandomForestRegressor(
    n_estimators=int(mejor["n_estimators"]),
    min_impurity_decrease=mejor["min_impurity_decrease"],
    random_state=RANDOM_STATE
)

modelo_final.fit(X_train, y_train)

pred_test = modelo_final.predict(X_test)

mse_test = mean_squared_error(
    y_test,
    pred_test
)

rmse_test = np.sqrt(mse_test)

print(f"MSE en test: {mse_test:,.2f}")
print(f"RMSE en test: {rmse_test:,.2f}")

MSE en test: 12,114,233,109.67
RMSE en test: 110,064.68


# PARTE 6 — TimeSeriesSplit

In [12]:
data_train_dev = data[
    data["Year"] <= 2021
].copy()

X_train_dev = data_train_dev.drop(
    columns=["Year", "Ventas_BMW_Unidades"]
)

y_train_dev = data_train_dev["Ventas_BMW_Unidades"]

In [13]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
tscv = TimeSeriesSplit(n_splits=4)

for i, (train_idx, val_idx) in enumerate(
    tscv.split(X_train_dev),
    start=1
):

    print(f"Bloque {i}")
    print(
        "Train:",
        data_train_dev.iloc[train_idx]["Year"].min(),
        "-",
        data_train_dev.iloc[train_idx]["Year"].max()
    )

    print(
        "Validación:",
        data_train_dev.iloc[val_idx]["Year"].min(),
        "-",
        data_train_dev.iloc[val_idx]["Year"].max()
    )

    print()

Bloque 1
Train: 2005 - 2009
Validación: 2010 - 2012

Bloque 2
Train: 2005 - 2012
Validación: 2013 - 2015

Bloque 3
Train: 2005 - 2015
Validación: 2016 - 2018

Bloque 4
Train: 2005 - 2018
Validación: 2019 - 2021



# PARTE 8 — GridSearchCV

In [14]:
rf = RandomForestRegressor(
    random_state=RANDOM_STATE
)

In [15]:
param_grid = {
    "n_estimators": [10, 25, 50, 100, 200, 300],
    "min_impurity_decrease": [
        0.0,
        1e6,
        5e6,
        1e7,
        5e7,
        1e8
    ]
}

In [16]:
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(
    X_train_dev,
    y_train_dev
)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=4, test_size=None),
             estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0, 1000000.0, 5000000.0,
                                                   10000000.0, 50000000.0,
                                                   100000000.0],
                         'n_estimators': [10, 25, 50, 100, 200, 300]},
             scoring='neg_root_mean_squared_error')

In [17]:
print("Mejores hiperparámetros:")
print(grid_search.best_params_)

print(
    "RMSE medio de validación:",
    -grid_search.best_score_
)

Mejores hiperparámetros:
{'min_impurity_decrease': 50000000.0, 'n_estimators': 10}
RMSE medio de validación: 235573.53278545538


# PARTE 9 — Evaluar este modelo en el mismo test

In [18]:
modelo_cv = grid_search.best_estimator_

modelo_cv.fit(
    X_train_dev,
    y_train_dev
)

pred_test_cv = modelo_cv.predict(X_test)

mse_test_cv = mean_squared_error(
    y_test,
    pred_test_cv
)

rmse_test_cv = np.sqrt(mse_test_cv)

print(f"MSE en test: {mse_test_cv:,.2f}")
print(f"RMSE en test: {rmse_test_cv:,.2f}")

MSE en test: 5,513,784,700.02
RMSE en test: 74,254.86


# PARTE 10 — Primera comparación

# PARTE 11 — Segunda versión: variaciones relativas

In [19]:
data_var = data.copy()

# Variación porcentual de las ventas de BMW
data_var["Ventas_BMW_Variacion"] = (
    data_var["Ventas_BMW_Unidades"].pct_change()
)

# Eliminar 2005, ya que no existe un año anterior
data_var = data_var.dropna(
    subset=["Ventas_BMW_Variacion"]
)

# Variables absolutas a transformar mediante variación porcentual
variables_variacion = [
    "Compensation_Per_Employee",
    "Industrial_Production",
    "Population_Total",
    "EUR_USD"
]

# Calcular la variación porcentual respecto al año anterior
for variable in variables_variacion:
    data_var[variable + "_Variacion"] = (
        data_var[variable].pct_change()
    )

# Eliminar las filas que hayan quedado con valores nulos
data_var = data_var.dropna()

# Mostrar las primeras observaciones
display(data_var.head())

# Mostrar dimensiones del dataset
print("Shape:", data_var.shape)

,Year,Ventas_BMW_Unidades,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD,Ventas_BMW_Variacion,Compensation_Per_Employee_Variacion,Industrial_Production_Variacion,Population_Total_Variacion,EUR_USD_Variacion
2,2007,1276793,32.75,145.5,437514477,1.3705,0.077382,0.025039,-0.014895,0.003379,0.091510
3,2008,1202239,33.86,148.0,439074280,1.4708,-0.058392,0.033893,0.017182,0.003565,0.073185
4,2009,1068770,34.41,144.5,440467898,1.3948,-0.111017,0.016243,-0.023649,0.003174,-0.051673
5,2010,1224280,35.20,127.2,441160594,1.3257,0.145504,0.022958,-0.119723,0.001573,-0.049541
6,2011,1380384,35.97,128.9,440526112,1.3920,0.127507,0.021875,0.013365,-0.001438,0.050011


Shape: (19, 11)


In [20]:
X_var = data_var[
    [
        "Compensation_Per_Employee_Variacion",
        "Industrial_Production_Variacion",
        "Population_Total_Variacion",
        "EUR_USD_Variacion"
    ]
]

y_var = data_var[
    "Ventas_BMW_Variacion"
]